# OPT-02 Spatial matching and fallback logic

This notebook focuses on the geography-matching logic used by `edges` once a method file and an inventory are already in place.


## Learning goals

After this notebook, you should be able to:

- explain why spatial matching is needed in regionalised LCIA
- distinguish direct, aggregate, dynamic, contained, and global fallback mapping
- predict which matching strategy should apply in simple geography cases
- identify the helper methods in `edges` that extend matching beyond direct hits


## Background references

- Sacchi, R., Menacho, A. H., Seitfudem, G., Agez, M., Schlesinger-Martinat, J., Koyamparambath, A., Saldivar, J. S., Loubet, P., & Bauer, C. (2025). Contextual LCIA without the overhead: an exchange-based framework for flexible impact assessment. *The International Journal of Life Cycle Assessment, 30*(12), 3087-3101. https://doi.org/10.1007/s11367-025-02551-7


## 1) Why matching logic is needed

A method file may define CFs for `FR`, `RER`, or `GLO`, while the inventory can contain exchanges located in `FR`, `ES`, `CA-QC`, or dynamic locations such as `RoW`. `edges` therefore needs a transparent sequence of matching rules.

In [ ]:
import json
from pathlib import Path

import pandas as pd
pd.set_option('display.max_colwidth', None)

example_2_path = Path('assets/edges_examples/lcia_example_2.json')

data = json.loads(example_2_path.read_text())

pd.DataFrame([
    {
        'supplier name': exc['supplier'].get('name'),
        'consumer location': exc['consumer'].get('location'),
        'value': exc['value'],
    }
    for exc in data['exchanges']
])


## 2) The main matching strategies in `edges`

The four main mapping strategies described in the questionnaire are complemented by a global fallback for the remaining eligible exchanges.  

|   | rule | example | idea |
|---|---|---|---|
| 0 | direct mapping | FR -> FR | The CF exists for the exact same location as the exchange. |
| 1 | aggregate-region<br>mapping | RER | A broader regional factor is computed based on CF of subregions that compose it. |
| 2 | dynamic-location<br>mapping | RoW -> average over<br>remaining country CFs | For dynamic locations such as RoW, `edges` starts from the country CFs available in the method and removes countries already represented explicitly in the inventory. |
| 3 | contained-location<br>mapping | CA-QC -> CA | A subregion inherits the factor of a containing region. |
| 4 | global fallback | XX -> GLO | A global factor is used when no more specific match can be applied. |

## Checkpoint 1

Predict which mapping strategy should apply in each row.  

|   | inventory location | available CF locations | predicted rule |
|---|---|---|---|
| 0 | FR | FR, RER, GLO |  |
| 1 | ES | RER, GLO |  |
| 2 | RoW | AU, BR, CA, DZ, GLO |  |
| 3 | CA-QC | CA, GLO |  |
| 4 | XX | GLO |  |


## 3) Which `edges` methods implement these steps?

The API mirrors the conceptual workflow quite closely. The diagram below is the usual sequence once `lci()` has built the inventory.  

|   | method | role |
|---|---|---|
| 0 | `map_exchanges()` | Apply the direct matching rules encoded in the method definition. |
| 1 | `map_aggregate_locations()` | Compute CFs for broader static regions such as RER, when exact matches are unavailable. |
| 2 | `map_dynamic_locations()` | Build CF for dynamic regions such as RoW or RoE from the remaining country CFs after explicit locations have been excluded. |
| 3 | `map_contained_locations()` | Use factors for containing regions, such as CA for CA-QC. |
| 4 | `map_remaining_locations_to_global()` | Assign a global fallback to the remaining eligible exchanges. |

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

steps = [
    'lci()',
    'map_exchanges()',
    'map_aggregate_\nlocations()',
    'map_dynamic_\nlocations()',
    'map_contained_\nlocations()',
    'map_remaining_\nlocations_to_global()',
]

fig, ax = plt.subplots(figsize=(14, 2.8))
ax.set_xlim(0, len(steps) * 2)
ax.set_ylim(0, 3)
ax.axis('off')

for i, step in enumerate(steps):
    x = 0.35 + i * 2
    box = FancyBboxPatch(
        (x, 1.2),
        1.45,
        0.7,
        boxstyle='round,pad=0.04,rounding_size=0.08',
        linewidth=1.4,
        edgecolor='#355C7D',
        facecolor='#E8F1F7',
    )
    ax.add_patch(box)
    ax.text(x + 0.725, 1.55, step, ha='center', va='center', fontsize=10)
    if i < len(steps) - 1:
        ax.annotate('', xy=(x + 1.95, 1.55), xytext=(x + 1.45, 1.55), arrowprops=dict(arrowstyle='->', lw=1.3))

ax.text(7, 0.55, '`generate_cf_table()` and `statistics()` are diagnostic tools used after the mapping steps.', ha='center', va='center', fontsize=10)
plt.title('Usual `edges` mapping sequence', pad=10)
plt.show()

## 4) Use `Geomatcher` to inspect aggregate and contained geographies

Before discussing dynamic `RoW`, it helps to isolate the easier cases:

- `ES -> RER` for an aggregate region
- `CA-QC -> CA` for a contained geography

In this section, you query `Geomatcher` directly to explain these static topology relationships.

In [ ]:
from contextlib import redirect_stderr, redirect_stdout
import io

from constructive_geometries import Geomatcher

geo = Geomatcher()


def _quiet_geo(method_name, location):
    sink = io.StringIO()
    try:
        with redirect_stdout(sink), redirect_stderr(sink):
            return list(
                getattr(geo, method_name)(
                    location,
                    include_self=False,
                    exclusive=False if method_name == 'within' else True,
                    biggest_first=False,
                )
            )
    except KeyError:
        return None


def _labels(items):
    return [item[1] if isinstance(item, tuple) else str(item) for item in items]


def parents_of(location, limit=12):
    result = _quiet_geo('within', location)
    return [] if result is None else _labels(result)[:limit]


def children_of(location, limit=12):
    result = _quiet_geo('contained', location)
    return [] if result is None else _labels(result)[:limit]


def is_known(location):
    return _quiet_geo('within', location) is not None

Start by looking at both directions of the topology query:

- `within(location)` gives broader regions that contain a location.
- `contained(location)` gives finer subregions contained in a region.

In [ ]:
pd.DataFrame([
    {
        'location': location,
        "within(location)": ', '.join(parents_of(location)) or '(none)',
        "contained(location)": ', '.join(children_of(location)) or '(none)',
    }
    for location in ['ES', 'CA-QC', 'BR', 'RER', 'CA']
])

The next map stays at the **country** level. It helps visualize the aggregate region `RER` and shows that example inventory locations like `ES` and `IT` sit inside that broader regional CF geography.

Two topology subtleties matter here:

- in `Geomatcher`, Spain is split so that `Canary Islands` is treated separately, so we re-add `ES` explicitly to make mainland Spain visible;
- `Russia (Europe)` is not a sovereign-country polygon, and `country_converter` maps it to `RUS`, which would incorrectly paint all of Russia.

For a country-level map, we therefore re-add `ES` but skip `Canary Islands` and `Russia (Europe)`.

In [ ]:
import country_converter as coco
import plotly.express as px

COUNTRY_LEVEL_EXCLUSIONS = {'Canary Islands', 'Russia (Europe)'}


def to_iso3(location):
    sink = io.StringIO()
    with redirect_stdout(sink), redirect_stderr(sink):
        iso3 = coco.convert(names=location, to='ISO3', not_found=None)
    if not iso3 or iso3 == location or len(str(iso3)) != 3:
        return None
    return str(iso3)


def plot_country_groups(groups, title):
    rows = []
    skipped = []
    for location, group in groups.items():
        if location in COUNTRY_LEVEL_EXCLUSIONS:
            skipped.append(f'{location} (cannot be drawn faithfully on a country-level map)')
            continue
        iso3 = to_iso3(location)
        if iso3 is None:
            skipped.append(location)
            continue
        rows.append({'location': location, 'iso3': iso3, 'group': group})
    frame = pd.DataFrame(rows)
    if frame.empty:
        print('No mappable country-level locations found.')
    else:
        fig = px.choropleth(
            frame,
            locations='iso3',
            color='group',
            hover_name='location',
            projection='natural earth',
            title=title,
        )
        fig.update_geos(showcoastlines=True, showcountries=True, fitbounds='locations')
        fig.show()
    if skipped:
        print('Not shown on the country-level map:', skipped)


rer_members = sorted(set(children_of('RER', limit=500)))
rer_members_for_map = sorted((set(rer_members) - COUNTRY_LEVEL_EXCLUSIONS) | {'ES'})
rer_map_groups = {
    location: ('Example inventory location inside RER' if location in {'ES', 'IT'} else 'Other RER member')
    for location in rer_members_for_map
}
plot_country_groups(rer_map_groups, 'Aggregate region example: RER and two sample inventory locations')

## Checkpoint 2

Use `constructive_geometries.Geomatcher` to fill in the missing answers.  

|   | location | question | your answer |
|---|---|---|---|
| 0 | ES | One broader region returned by `within(ES)` |  |
| 1 | CA-QC | Containing country used to explain CA-QC |  |
| 2 | BR | Is BR inside RER? (True/False) |  |
| 3 | XX | Is XX known to Geomatcher? (True/False) |  |

In [ ]:
from constructive_geometries import Geomatcher
# from edges.georesolver import Geomatcher this works too since `edges` exposes Geomatcher

In [ ]:
geo = Geomatcher()

In [ ]:
geo.within("ES")

In [ ]:
geo.within("CA-QC")

In [ ]:
"BR" in geo.contained(("ecoinvent", "RER"))

In [ ]:
geo.contained(("ecoinvent", "RER"))

In [ ]:
"XX" in geo.keys()

In [ ]:
# we can list all entitites known by `Geomatcher`
list(geo.keys())

## 5) Dynamic `RoW` on a real BAFU activity

`RoW` is different from `RER` or `CA`. It is not a fixed region in the method file. Instead, `edges` builds it dynamically from the **country CFs** that remain available after excluding same-product locations already present in the inventory.

Here we use `Natural gas, burned in gas turbine` at location `RoW` for two complementary views:

- the **actual** `edges` behavior with a country-level teaching method;
- a **counterfactual teaching view** where we ignore aggregate inventory locations and ask a simpler question: which countries would remain in `RoW` if we excluded only the explicit same-product country datasets?

This teaching simplification matches the structure of `edges` method files, which only contain country CFs here.


In [ ]:
import json
import tempfile
from pathlib import Path

import bw2data as bd
import numpy as np
from IPython.display import display
from edges import EdgeLCIA

Start by locating the activity and listing the same-product locations already present in the inventory.

In [ ]:
bd.projects.set_current('paris-lca-course-2026')
db_name = 'bafu'

db = bd.Database(db_name)

bafu_activity = [
        act for act in db
        if act['name'].startswith("Natural gas, burned in gas turbine")
        and act.get('location') == 'RoW'
][0]


In [ ]:
bafu_activity.as_dict()

In [ ]:
for e in bafu_activity.exchanges():
    print(e["name"], " | ", e.get("product"), " | ", e["amount"], " | ", e["unit"], " | ",  e["type"])

Let's build a fake method that contains a CF for all the cuntries in the world.

Two different interpretations are now useful:

1. **Actual `edges` behavior**: because `GLO` is one of the same-product locations in the inventory, the dynamic `RoW` pool can collapse.
2. **Counterfactual teaching view**: if we ignore `GLO` just for the map, we can still visualize the distinction between countries with explicit datasets and countries that would otherwise remain in `RoW`.

The next cells show both on purpose.

In [ ]:
import country_converter as coco
world_country_codes = [
    code
    for code in sorted(coco.CountryConverter().data['ISO2'].dropna().astype(str).unique()) if len(code) == 2 
]
world_country_codes

In [ ]:
country_factors = {code: {
        "value": np.random.randint(1, 100),
        "weight": np.random.randint(1, 100),
    }
    for code in world_country_codes
}


dynamic_method = {
    'name': 'Dynamic RoW teaching example',
    'version': '1.0',
    'unit': 'kg CO2-eq',
    'description': 'Teaching method with one country CF for every country.',
    'exchanges': [
        {
            'supplier': {'name': 'Carbon dioxide, fossil', 'matrix': 'biosphere'},
            'consumer': {'matrix': 'technosphere', 'location': loc},
            'value': dictionary["value"],
            'weight': dictionary["weight"],
        }
        for loc, dictionary in country_factors.items()
    ],
}
dynamic_method

Now let's run `edges` with that method and see if `RoW` exchanges get a CF

In [ ]:
bafu_activity

In [ ]:
row_lca = EdgeLCIA(
    demand={bafu_activity: 1},
    method=dynamic_method,
)
row_lca.lci()
row_lca.map_exchanges()
row_lca.map_aggregate_locations()
row_lca.map_dynamic_locations()
row_lca.evaluate_cfs()
row_lca.lcia()

In [ ]:
cf_table = row_lca.generate_cf_table(include_unmatched=True)
gas_turbine_carbon_row = cf_table[
    (cf_table["supplier name"] == "Carbon dioxide, fossil")
    & (cf_table["consumer name"] == bafu_activity["name"])
    & (cf_table["consumer reference product"] == bafu_activity["reference product"])
    & (cf_table["consumer location"] == "RoW")
].iloc[0]

In [ ]:
gas_turbine_carbon_row[[
    "supplier name",
    "consumer name",
    "consumer location",
    "CF",
    "impact",
]].to_frame().T

In [ ]:
same_product_family_locations = sorted({
    act["location"]
    for act in db
    if act["name"] == bafu_activity["name"]
    and act["reference product"] == bafu_activity["reference product"]
})
explicit_country_locations = [
    loc for loc in same_product_family_locations if loc in world_country_codes
]
aggregate_locations = [
    loc for loc in same_product_family_locations if loc not in explicit_country_locations
]
dynamic_row_pool_counterfactual = [
    country for country in world_country_codes if country not in explicit_country_locations
]

In [ ]:
row_map_groups = {
    **{loc: 'Explicit same-product country dataset already in the inventory' for loc in explicit_country_locations},
    **{loc: 'Remaining country in the teaching RoW pool' for loc in dynamic_row_pool_counterfactual},
}
plot_country_groups(
    row_map_groups,
    'Teaching map for Natural gas, burned in gas turbine: explicit country datasets and remaining RoW pool'
)

We can also manually reproduce the `RoW` CF for this exact gas-turbine dataset by looping through the JSON method entries for the countries that belong to `RoW` here.

`edges` stores the deterministic composite CF as a symbolic expression with normalized shares rounded to **four decimals**, so we might not land exactly on the same value.

In [ ]:
selected_json_cfs = [
    exc
    for exc in dynamic_method["exchanges"]
    if exc["supplier"]["name"] == "Carbon dioxide, fossil"
    and exc["consumer"]["location"] in dynamic_row_pool_counterfactual
]

weights = np.array([exc["weight"] for exc in selected_json_cfs], dtype=float)
normalized_shares = weights / weights.sum()
rounded_shares = np.round(normalized_shares, 4)
manual_row_cf = float(sum(
    share * exc["value"]
    for share, exc in zip(rounded_shares, selected_json_cfs)
))

np.testing.assert_allclose(manual_row_cf, float(gas_turbine_carbon_row["CF"]), rtol=1e-2)

{
    "countries in RoW for this dataset": len(gas_turbine_row_countries),
    "manual CF from the JSON method": manual_row_cf,
    "CF from edges": float(gas_turbine_carbon_row["CF"]),
    "difference": manual_row_cf - float(gas_turbine_carbon_row["CF"]),
}

## Recap

After this notebook, you should now be able to:

- explain why spatial matching is needed in `edges`
- connect each conceptual matching rule to the corresponding helper method in the API
- query `Geomatcher` directly to explain aggregate and contained geographies
- explain why dynamic `RoW` is computed from the remaining country CFs instead of read as a fixed factor
- distinguish a real run where `RoW` collapses because `GLO` is present from a counterfactual map that ignores `GLO` for teaching purposes
- use simple maps to explain aggregate regions and dynamic-country pools